# 03. Cross-Language Transfer Matrix

**Paper section:** §5 Cross-language transfer (Table 3).
**What it computes:** Computes the 3 × 3 transfer matrix: train transitional-probability segmenter on language A, evaluate on language B. Reports F1 with bootstrap confidence intervals so the script-vs-language contribution can be read off the matrix directly. Replicates Table 3 of the paper.
**Inputs:** Outputs of notebook 01 (token-level dataframes per language).
**Outputs:** `outputs/table3_transfer_matrix.csv` (F1 matrix), `outputs/table3_transfer_matrix_ci.csv` (bootstrap CIs), `outputs/figure2_transfer_bars.png`.
**Expected runtime (CPU baseline):** ~10 min on CPU (bootstrap dominates).

All randomness uses `SEED = 42`.

## Setup
This notebook uses shared utilities from `_setup.py`:
- `load_corpora()`: returns dict of dataframes per language
- `POS_HARMONIZATION`: dict mapping raw POS tags to unified tagset
- `BASE_PATH`: data directory (set this for your environment)

```python
from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH
```


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
try:
    import torch
    torch.manual_seed(42)
except ImportError:
    pass

from _setup import load_corpora, POS_HARMONIZATION, BASE_PATH, set_seeds
set_seeds(42)


In [ ]:
# --- data-availability guard ------------------------------------------
missing = [l for l in ('akk', 'sux', 'elx') if l not in corpora]
if missing:
    print(f'WARNING: {missing} not loaded. Cells that depend on them will be skipped.')
    print(f'Set $CUNEI_DATA to a directory containing alltexts_AKK.csv, alltexts_SUX.csv, and the Elamite files.')
# Convenience: expose datasets dict for cells originally from the monolith.
datasets = {l: corpora[l] for l in ('akk', 'sux', 'elx') if l in corpora}
sign_dict = corpora.get('_sign_dict', {})


In [ ]:
corpora = load_corpora(BASE_PATH)
doc_corpora = corpora['_documents']


## Cross Language Transfer

In [ ]:
# ============================================================
# HITTITE CROSS-LANGUAGE TRANSFER
# ============================================================
# Paste this cell into ThreeLanguagePipeline_v2 AFTER cells 1-17
# (i.e., after data loading & sign list setup).
# Requires: sign_dict, doc_corpora (with akk/sux/elx) already loaded.
# ============================================================

import unicodedata
import random
from collections import Counter

# ── 1. Hittite-specific preprocessing & Unicode conversion ──

HITT_PATH = BASE_PATH + '7000_hitt_txts_wGloss.csv'

def normalize_for_lookup_hitt(tok):
    """Normalize Hittite characters for sign list lookup."""
    t = tok
    t = t.replace('ḫ', 'h').replace('Ḫ', 'H')
    t = ''.join(c for c in unicodedata.normalize('NFD', t)
                if unicodedata.category(c) != 'Mn')
    t = t.replace('~', '').replace('˽', '')
    t = t.replace('⸢', '').replace('⸣', '')
    return t

def lookup_sign_hitt(tok):
    """Try multiple normalization strategies."""
    for t in [tok, tok.lower(), tok.upper()]:
        if t in sign_dict: return sign_dict[t]
    n = normalize_for_lookup_hitt(tok)
    for t in [n, n.lower(), n.upper()]:
        if t in sign_dict: return sign_dict[t]
    stripped = re.sub(r'[₀₁₂₃₄₅₆₇₈₉]+$', '', n)
    for t in [stripped, stripped.lower(), stripped.upper()]:
        if t in sign_dict: return sign_dict[t]
    stripped2 = re.sub(r'\d+$', '', n)
    for t in [stripped2, stripped2.lower(), stripped2.upper()]:
        if t in sign_dict: return sign_dict[t]
    return None

def hittite_translit_to_signs(translit):
    """Convert Hittite transliteration to list of Unicode signs.

    Preprocessing per Zenodo documentation:
    1. Replace { } with whitespace
    2. Remove [ ] and ⸢ ⸣ (no whitespace)
    3. Convert to Unicode signs
    """
    t = str(translit)
    t = t.replace('{', ' ').replace('}', ' ')
    t = t.replace('[', '').replace(']', '')
    t = t.replace('⸢', '').replace('⸣', '')
    t = t.replace('-', ' ').replace('.', ' ')
    for ch in ['#', '!', '?', '*', '(', ')', '°', '½']:
        t = t.replace(ch, '')
    t = re.sub(r'\s+', ' ', t).strip()

    signs = []
    for tok in t.split():
        tok = tok.strip()
        if not tok: continue
        uni = lookup_sign_hitt(tok)
        if uni and str(uni) != 'nan':
            signs.append(uni)
    return signs

# ── 2. Load Hittite data ──

print("Loading Hittite data...")
hitt_raw = pd.read_csv(HITT_PATH)
hitt_raw = hitt_raw[hitt_raw['translit'].notna() & (hitt_raw['translit'] != '…')].copy()
print(f"  Raw tokens: {len(hitt_raw):,}")
print(f"  Texts: {hitt_raw['txtid'].nunique():,}")

# Convert to Unicode and build segmented documents
hitt_docs_segmented = {}
total_words = 0
total_converted = 0

for txtid, group in hitt_raw.groupby('txtid'):
    words = []
    for _, row in group.iterrows():
        signs = hittite_translit_to_signs(row['translit'])
        if signs:
            words.append(signs)
            total_converted += 1
        total_words += 1
    if len(words) >= 2:
        hitt_docs_segmented[txtid] = words

print(f"  Segmented documents: {len(hitt_docs_segmented):,}")
print(f"  Words converted: {total_converted:,}/{total_words:,} ({total_converted/total_words*100:.1f}%)")
total_signs = sum(sum(len(w) for w in d) for d in hitt_docs_segmented.values())
unique_signs = set(s for d in hitt_docs_segmented.values() for w in d for s in w)
print(f"  Total signs: {total_signs:,}")
print(f"  Unique Unicode signs: {len(unique_signs)}")

# Also build Unicode doc strings for doc_corpora compatibility
hitt_docs_unicode = {}
for txtid, group in hitt_raw.groupby('txtid'):
    unicode_words = []
    for _, row in group.iterrows():
        signs = hittite_translit_to_signs(row['translit'])
        if signs:
            unicode_words.append(' '.join(signs))
    if unicode_words:
        hitt_docs_unicode[txtid] = ' '.join(unicode_words)

# Build Latin doc strings
hitt_docs_latin = {}
for txtid, group in hitt_raw.groupby('txtid'):
    hitt_docs_latin[txtid] = ' '.join(group['translit'].dropna().astype(str))

doc_corpora['hit'] = {'latin': hitt_docs_latin, 'unicode': hitt_docs_unicode}
print(f"\n  Added 'hit' to doc_corpora: {len(hitt_docs_latin)} docs")

# ── 3. Build segmented docs for AKK/SUX/ELX ──
# (same format as Hittite: dict of {doc_id: [[sign, sign], [sign, sign, sign], ...]})

def unicode_docs_to_segmented(unicode_doc_dict):
    """Convert space-separated Unicode doc strings to segmented format."""
    segmented = {}
    for tid, doc in unicode_doc_dict.items():
        if not doc or not doc.strip():
            continue
        words = doc.split()
        word_signs = []
        for word in words:
            signs = [ch for ch in word if ch.strip()]
            if signs:
                word_signs.append(signs)
        if len(word_signs) >= 2:
            segmented[tid] = word_signs
    return segmented

all_segmented = {}
for lang in ['akk', 'sux', 'elx']:
    if lang in doc_corpora:
        all_segmented[lang] = unicode_docs_to_segmented(doc_corpora[lang]['unicode'])
        print(f"  {lang.upper()}: {len(all_segmented[lang])} segmented docs")

all_segmented['hit'] = hitt_docs_segmented
print(f"  HIT: {len(all_segmented['hit'])} segmented docs")

# ── 4. TP functions ──

def compute_tp(doc_ids, docs):
    bi, uni = Counter(), Counter()
    for did in doc_ids:
        s = [sign for w in docs[did] for sign in w]
        for i in range(len(s)):
            uni[s[i]] += 1
            if i < len(s) - 1:
                bi[(s[i], s[i+1])] += 1
    return {k: c / uni[k[0]] for k, c in bi.items()}

def evaluate_tp(doc_ids, docs, tp, theta):
    tc = fc = fnc = 0
    for did in doc_ids:
        words = docs[did]
        s = [sign for w in words for sign in w]
        gold = set(); pos = 0
        for w in words:
            pos += len(w)
            gold.add(pos)
        gold.discard(pos)
        pred = {i+1 for i in range(len(s)-1)
                if tp.get((s[i], s[i+1]), 0) < theta}
        tc += len(pred & gold)
        fc += len(pred - gold)
        fnc += len(gold - pred)
    p = tc / (tc + fc) if (tc + fc) else 0
    r = tc / (tc + fnc) if (tc + fnc) else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    return f1, p, r

def find_best_threshold(doc_ids, docs, tp):
    best_f1, best_theta = 0, 0.5
    for theta in np.arange(0.05, 0.99, 0.05):
        f1, _, _ = evaluate_tp(doc_ids, docs, tp, theta)
        if f1 > best_f1:
            best_f1, best_theta = f1, theta
    for theta in np.arange(max(0.01, best_theta - 0.1),
                            min(0.99, best_theta + 0.1), 0.01):
        f1, _, _ = evaluate_tp(doc_ids, docs, tp, theta)
        if f1 > best_f1:
            best_f1, best_theta = f1, theta
    f1, p, r = evaluate_tp(doc_ids, docs, tp, best_theta)
    return {'f1': f1, 'precision': p, 'recall': r, 'threshold': best_theta}

# ── 5. Full 4×4 cross-language transfer ──

lang_names = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite', 'hit': 'Hittite'}
languages = [l for l in ['akk', 'sux', 'elx', 'hit'] if l in all_segmented]

print(f"\n{'='*70}")
print(f"  CROSS-LANGUAGE WORD BOUNDARY TRANSFER (4 languages)")
print(f"{'='*70}")
print(f"\n  {'Train':>12s} {'Test':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ*':>6s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")

transfer_results = {}

for train_lang in languages:
    train_docs = all_segmented[train_lang]
    train_ids = list(train_docs.keys())
    tp = compute_tp(train_ids, train_docs)

    for test_lang in languages:
        test_docs = all_segmented[test_lang]
        test_ids = list(test_docs.keys())
        metrics = find_best_threshold(test_ids, test_docs, tp)

        key = f"{train_lang}→{test_lang}"
        transfer_results[key] = metrics

        marker = "  ← same" if train_lang == test_lang else ""
        beat = ""
        if train_lang != test_lang:
            same_f1 = transfer_results.get(f"{test_lang}→{test_lang}", {}).get('f1', 0)
            if same_f1 > 0 and metrics['f1'] > same_f1:
                beat = "  ★ BEATS SAME-LANG"

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{metrics['f1']:>8.4f} {metrics['precision']:>8.4f} "
              f"{metrics['recall']:>8.4f} {metrics['threshold']:>6.2f}{marker}{beat}")

# ── 6. Zero-shot transfer (source threshold, no tuning) ──

print(f"\n{'='*70}")
print(f"  ZERO-SHOT TRANSFER (source language θ, no tuning on test)")
print(f"{'='*70}")
print(f"\n  {'Train':>12s} {'Test':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ(src)':>8s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")

for train_lang in languages:
    src_theta = transfer_results[f"{train_lang}→{train_lang}"]['threshold']
    train_docs = all_segmented[train_lang]
    tp = compute_tp(list(train_docs.keys()), train_docs)

    for test_lang in languages:
        if test_lang == train_lang:
            continue
        test_docs = all_segmented[test_lang]
        f1, p, r = evaluate_tp(list(test_docs.keys()), test_docs, tp, src_theta)

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{f1:>8.4f} {p:>8.4f} {r:>8.4f} {src_theta:>8.2f}")

# ── 7. Summary matrix ──

print(f"\n{'='*70}")
print(f"  TRANSFER MATRIX (F1)")
print(f"{'='*70}")

train_test_label = "Train\\Test"
header = f"  {train_test_label:>12s}"
for tl in languages:
    header += f" {tl.upper():>8s}"
print(header)
print(f"  {'-'*12}" + f" {'-'*8}" * len(languages))

for trl in languages:
    row = f"  {trl.upper():>12s}"
    for tl in languages:
        key = f"{trl}→{tl}"
        f1 = transfer_results[key]['f1']
        row += f" {f1:>8.4f}"
    print(row)

# ── 8. Key findings ──

print(f"\n{'='*70}")
print(f"  KEY FINDINGS")
print(f"{'='*70}")

# Mesopotamian → Hittite
for src in ['akk', 'sux', 'elx']:
    key = f"{src}→hit"
    hit_same = transfer_results['hit→hit']['f1']
    cross = transfer_results[key]['f1']
    delta = cross - hit_same
    print(f"  {lang_names[src]:>12s} → Hittite: F1={cross:.4f} "
          f"(vs HIT→HIT {hit_same:.4f}, Δ={delta:+.4f})")

# Hittite → Mesopotamian
print()
for tgt in ['akk', 'sux', 'elx']:
    key = f"hit→{tgt}"
    same_key = f"{tgt}→{tgt}"
    same_f1 = transfer_results[same_key]['f1']
    cross = transfer_results[key]['f1']
    delta = cross - same_f1
    print(f"  Hittite → {lang_names[tgt]:>12s}: F1={cross:.4f} "
          f"(vs same-lang {same_f1:.4f}, Δ={delta:+.4f})")


In [ ]:
# ============================================================
# CROSS-LANGUAGE WORD BOUNDARY TRANSFER
# ============================================================
# Train TP statistics on one cuneiform language,
# test on another. If it works, word boundaries are
# a property of the SCRIPT, not the LANGUAGE.
#
# Prerequisites: cells 1-13 (data loading)
# ============================================================

!pip install git+https://anonymous.4open.science/r/cunei-tools

from cunei_tools import CuneiSeg
import numpy as np

# Collect documents per language
all_docs = {}
for lang in ['akk', 'sux', 'elx']:
    docs = list(doc_corpora[lang]['unicode'].values())
    docs = [d for d in docs if d and len(d.split()) > 2]
    all_docs[lang] = docs
    print(f"{lang.upper()}: {len(docs)} documents")

# ============================================================
# Experiment 1: Full cross-language transfer
# Train on language A, test on language B
# ============================================================
print(f"\n{'='*60}")
print(f"  CROSS-LANGUAGE WORD BOUNDARY TRANSFER")
print(f"{'='*60}")

lang_names = {'akk': 'Akkadian', 'sux': 'Sumerian', 'elx': 'Elamite'}

print(f"\n  {'Train →':>12s} {'Test →':>12s} {'F1':>8s} {'P':>8s} {'R':>8s} {'θ*':>6s}")
print(f"  {'-'*12} {'-'*12} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")

transfer_results = {}

for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        seg = CuneiSeg()
        seg.train(all_docs[train_lang])
        metrics = seg.find_optimal_threshold(all_docs[test_lang])

        key = f"{train_lang}→{test_lang}"
        transfer_results[key] = metrics

        marker = "  (same)" if train_lang == test_lang else ""
        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{metrics['f1']:>8.4f} {metrics['precision']:>8.4f} "
              f"{metrics['recall']:>8.4f} {metrics['threshold']:>6.2f}{marker}")

# ============================================================
# Experiment 2: Transfer with FIXED threshold
# Train on A with A's optimal threshold, apply directly to B
# No threshold tuning on the test language
# ============================================================
print(f"\n{'='*60}")
print(f"  ZERO-SHOT TRANSFER (fixed threshold)")
print(f"{'='*60}")

# First get optimal thresholds per language
optimal_thresholds = {}
for lang in ['akk', 'sux', 'elx']:
    seg = CuneiSeg()
    seg.train(all_docs[lang])
    m = seg.find_optimal_threshold(all_docs[lang])
    optimal_thresholds[lang] = m['threshold']
    print(f"  {lang.upper()} optimal θ = {m['threshold']:.2f}")

print(f"\n  {'Train →':>12s} {'Test →':>12s} {'θ (from train)':>14s} {'F1':>8s}")
print(f"  {'-'*12} {'-'*12} {'-'*14} {'-'*8}")

for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        if train_lang == test_lang:
            continue

        seg = CuneiSeg()
        seg.train(all_docs[train_lang])

        # Use train language's threshold, no tuning on test
        theta = optimal_thresholds[train_lang]

        # Evaluate manually with fixed threshold
        tp_total, fp_total, fn_total = 0, 0, 0
        for doc in all_docs[test_lang]:
            words = doc.split()
            continuous = doc.replace(' ', '')
            if len(continuous) < 3 or len(words) < 2:
                continue

            # Gold boundaries
            gold = set()
            pos = 0
            for w in words:
                pos += len(w)
                gold.add(pos)
            gold.discard(len(continuous))  # remove end

            # Predicted boundaries
            pred = set()
            for i in range(1, len(continuous)):
                bigram = (continuous[i-1], continuous[i])
                uni = seg.unigrams.get(continuous[i-1], 0)
                bi = seg.bigrams.get(bigram, 0)
                if uni > 0:
                    tp_val = bi / uni
                    if tp_val < theta:
                        pred.add(i)

            tp_total += len(gold & pred)
            fp_total += len(pred - gold)
            fn_total += len(gold - pred)

        p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0
        r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0
        f1 = 2 * p * r / (p + r) if (p + r) else 0

        print(f"  {lang_names[train_lang]:>12s} {lang_names[test_lang]:>12s} "
              f"{theta:>14.2f} {f1:>8.4f}")

# ============================================================
# Summary
# ============================================================
print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")

print(f"\n  Same-language (baseline):")
for lang in ['akk', 'sux', 'elx']:
    key = f"{lang}→{lang}"
    print(f"    {lang.upper()}: F1 = {transfer_results[key]['f1']:.4f}")

print(f"\n  Cross-language transfer (with threshold tuning on test):")
for train_lang in ['akk', 'sux', 'elx']:
    for test_lang in ['akk', 'sux', 'elx']:
        if train_lang == test_lang:
            continue
        key = f"{train_lang}→{test_lang}"
        same_key = f"{test_lang}→{test_lang}"
        drop = transfer_results[same_key]['f1'] - transfer_results[key]['f1']
        print(f"    {train_lang.upper()}→{test_lang.upper()}: "
              f"F1 = {transfer_results[key]['f1']:.4f} "
              f"(Δ = {-drop:+.4f} vs same-lang)")